# Day 2 — Pandas & Export CSV

Goal: Load raw CSV into pandas, clean basics, and export clean_data.csv

## Setup
If you don't have the required packages, run in a terminal:
```bash
pip install pandas
```


In [1]:
# Load raw_data.csv (created by the scraper or created as a sample if missing)
import pandas as pd
from pathlib import Path

raw_path = Path('raw_data.csv')

# If the CSV doesn't exist yet, create a tiny sample so the notebook can run
if not raw_path.exists():
    sample = [
        {
            'scraped_at': '2026-06-30T12:00:00',
            'item': 'Croissant',
            'price_text': 'CHF 2.50',
            'price': None,
            'quantity': None,
            'store': 'Main',
            'url': 'https://example.com/croissant'
        },
        {
            'scraped_at': '2026-06-30T12:05:00',
            'item': 'Baguette',
            'price_text': 'CHF 3.00',
            'price': None,
            'quantity': 2,
            'store': 'Main',
            'url': 'https://example.com/baguette'
        }
    ]
    pd.DataFrame(sample).to_csv(raw_path, index=False, encoding='utf-8')
    print('Wrote sample raw_data.csv — replace with your real raw_data.csv from Day 1')

# Read the CSV. If your CSV uses a different separator (e.g., ';') add sep=';' to read_csv.
# parse_dates will try to convert the scraped_at column to datetime
df = pd.read_csv(raw_path, parse_dates=['scraped_at'], encoding='utf-8')

# Quick check
print('Loaded rows:', len(df))
display(df.head())

Wrote sample raw_data.csv — replace with your real raw_data.csv from Day 1
Loaded rows: 2


,scraped_at,item,price_text,price,quantity,store,url
0,2026-06-30 12:00:00,Croissant,CHF 2.50,NaN,NaN,Main,https://example.com/croissant
1,2026-06-30 12:05:00,Baguette,CHF 3.00,NaN,2.0,Main,https://example.com/baguette


## Inspect the DataFrame
Run the checks below to see types and missing values.


In [2]:
print('shape:', df.shape)
print(df.info())
print('Missing values per column:')
print(df.isna().sum())

shape: (2, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   scraped_at  2 non-null      datetime64[ns]
 1   item        2 non-null      object        
 2   price_text  2 non-null      object        
 3   price       0 non-null      float64       
 4   quantity    1 non-null      float64       
 5   store       2 non-null      object        
 6   url         2 non-null      object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 240.0+ bytes
None
Missing values per column:
scraped_at    0
item          0
price_text    0
price         2
quantity      1
store         0
url           0
dtype: int64


## Cleaning steps
- Parse scraped_at to datetime
- Ensure price is numeric: use price if present, else try to parse price_text
- Fill missing quantity with 1 and convert to int
- Compute total = price * quantity
- Drop rows missing item or price


In [3]:
import re
from datetime import datetime

def parse_price_text(s):
    if s is None:
        return None
    try:
        # remove currency letters and spaces, replace commas with dots
        s2 = str(s).replace(' ','').replace('CHF','').replace('€','').replace('$','').strip()
        s2 = s2.replace(',', '.')
        # keep digits and dot
        s2 = ''.join(ch for ch in s2 if (ch.isdigit() or ch == '.'))
        return float(s2) if s2 else None
    except Exception:
        return None

# Parse scraped_at
df['scraped_at'] = pd.to_datetime(df.get('scraped_at'), errors='coerce')

# Ensure price numeric
if 'price' in df.columns:
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
else:
    df['price'] = None

if df['price'].isna().any() and 'price_text' in df.columns:
    df['price'] = df['price'].fillna(df['price_text'].apply(parse_price_text))

# Fill quantity
df['quantity'] = df.get('quantity').fillna(1).astype(int)

# Compute total
df['total'] = df['price'] * df['quantity']

# Drop bad rows
before = len(df)
df = df[df['item'].notna() & df['price'].notna()]
after = len(df)
print(f'dropped {before-after} rows missing item or price')

df.head()


dropped 0 rows missing item or price


,scraped_at,item,price_text,price,quantity,store,url,total
0,2026-06-30 12:00:00,Croissant,CHF 2.50,2.5,1,Main,https://example.com/croissant,2.5
1,2026-06-30 12:05:00,Baguette,CHF 3.00,3.0,2,Main,https://example.com/baguette,6.0


## Deduplicate and export CSV
Create a dedupe key (url + item + scraped_date) and drop duplicates before writing CSV.


In [4]:
# Create dedupe key
df['scraped_date'] = df['scraped_at'].dt.date
df['dedupe_key'] = df.get('url', '').astype(str) + '|' + df['item'].astype(str) + '|' + df['scraped_date'].astype(str)
df = df.drop_duplicates(subset=['dedupe_key'])

out_path = Path('sales_clean.csv')
df.to_csv(out_path, index=False)
print('Wrote', len(df), 'rows to', out_path)


Wrote 2 rows to sales_clean.csv
